# Xarray : températures de surface de la mer

Ce TP explore avec xarray des données climatiques au format NetCDF : la température de surface de la mer sur tout le globe, mois par mois, de 1970 à 2021. Vous lirez le fichier, sélectionnerez un point et une région par leurs coordonnées, calculerez la climatologie et les anomalies qui révèlent El Niño, puis écrirez un résultat en NetCDF et en Zarr, bloc par bloc, avant de regarder le HDF5 qui se cache sous le NetCDF.

Les exercices s'enchaînent : chacun reprend des résultats des précédents, que leurs solutions calculent. Le dernier, une animation, est facultatif.

## Installation

Colab n'a pas Zarr : la cellule suivante l'installe. xarray, Dask, h5py et h5netcdf y sont déjà.

In [ ]:
!pip install -q zarr

## Imports

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

## Les données

Le jeu *Extended Reconstructed Sea Surface Temperature* (ERSST), version 5, de l'agence américaine NOAA reconstruit la température de surface de la mer chaque mois depuis 1854, sur une grille de 2° × 2°, à partir des mesures des navires et des bouées ([présentation](https://www.ncei.noaa.gov/products/extended-reconstructed-sst)). Le module `xarray.tutorial` en fournit un extrait, de 1970 à 2021 : un fichier NetCDF de 19 Mo, téléchargé au premier appel puis gardé en cache. Une valeur est la température moyenne d'un mois dans une maille de la grille, en degrés Celsius.

`open_dataset` renvoie un `Dataset` ; sa variable `sst` (*sea surface temperature*), un `DataArray`, est celle de tout le TP.

In [ ]:
ds = xr.tutorial.open_dataset("ersstv5")
sst = ds.sst
ds

## 1. Explorer le fichier NetCDF

L'affichage du `Dataset`, ci-dessus, résume le fichier. Répondez aux questions suivantes, en écrivant le code qui montre chaque réponse :

- quelles sont les dimensions de `sst`, et combien de valeurs contient-elle ?
- quelle période couvre-t-elle, et quel est le pas de la grille, en degrés ?
- dans quelle unité est la température (les attributs de `sst`) ?
- quelle part des mailles d'une carte n'a pas de valeur, et pourquoi ?
- comment le fichier range-t-il `sst` sur le disque : est-elle compressée, et en blocs de quelle forme (son `encoding`) ? Dans quelle unité les dates sont-elles écrites dans le fichier ?

Tracez enfin la carte du premier mois avec `.plot()`.

**Documentation utile**

- [`DataArray.isnull`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.isnull.html)
- [Lire les données encodées](https://docs.xarray.dev/en/stable/user-guide/io.html#reading-encoded-data)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`sst.sizes` donne la taille de chaque dimension, `sst.size` le nombre de valeurs. `ds.time.values[[0, -1]]` donne la première et la dernière date ; la différence de deux latitudes voisines, le pas. `sst.attrs` est un dictionnaire, comme `sst.encoding` (clés `zlib` et `chunksizes`) et `ds.time.encoding` (clé `units`). `.isnull()` renvoie un tableau de booléens, dont la moyenne est la proportion de valeurs manquantes : un peu moins d'un tiers.

```python
print(sst.sizes, sst.size)
print(ds.time.values[[0, -1]])
print(float(ds.lat[0] - ds.lat[1]), float(...))
print(sst.attrs[...])
premiere_carte = sst.isel(time=0)
print(float(premiere_carte.isnull().mean()))
print(sst.encoding[...], sst.encoding[...], ds.time.encoding[...])
premiere_carte.plot()
```

### Solution

In [ ]:
print(sst.sizes, sst.size)
print(ds.time.values[[0, -1]])
print(float(ds.lat[0] - ds.lat[1]), float(ds.lon[1] - ds.lon[0]))
print(sst.attrs["units"], sst.attrs["long_name"])
premiere_carte = sst.isel(time=0)
print(float(premiere_carte.isnull().mean()))
print(sst.encoding["zlib"], sst.encoding["chunksizes"], ds.time.encoding["units"])
premiere_carte.plot()
# 624 mois × 89 latitudes × 180 longitudes : 9 996 480 valeurs.
# De janvier 1970 à décembre 2021, sur une grille de 2° en latitude et en longitude.
# En degrés Celsius (degC) : des moyennes mensuelles.
# 31 % des mailles n'ont pas de valeur : les terres, et la banquise, sans mer à mesurer.
# Compressée (zlib), en blocs d'une carte, (1, 89, 180) : lire un mois ne lit qu'un bloc.
# Les dates sont écrites en jours depuis le 1er janvier 1800 : xarray les décode.

## 2. Sélectionner par étiquette

Avec `sel`, extrayez :

- la série des températures à Brest (48,4° N, 4,5° O), dans la maille la plus proche. Tracez-la, puis calculez la température moyenne de chaque mois de l'année (`groupby("time.month")`) : quel est le mois le plus froid, le plus chaud, et quel écart les sépare ?
- la région Niño 3.4 du Pacifique équatorial, de 5° S à 5° N et de 170° O à 120° O, puis sa température moyenne de chaque mois, sur toutes ses mailles : la série `nino34`, que les exercices suivants utilisent.

Attention : les longitudes du fichier vont de 0 à 360 vers l'est, et ses latitudes sont rangées du nord au sud.

**Documentation utile**

- [`DataArray.sel`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sel.html)
- [`DataArray.idxmax`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.idxmax.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une longitude ouest $x$ s'écrit $360 - x$ : 4,5° O devient 355,5, et 170° O devient 190. `sel(lat=..., lon=..., method="nearest")` choisit la maille la plus proche. Une tranche `slice(debut, fin)` suit l'ordre de la coordonnée : sur des latitudes décroissantes, `slice(-5, 5)` renvoie un tableau vide, sans erreur. `idxmin()` et `idxmax()` donnent l'étiquette (ici, le mois) du minimum et du maximum. `mean(["lat", "lon"])` fait la moyenne sur les deux dimensions à la fois. À Brest, les mois les plus froid et le plus chaud sont séparés de 7 à 8 °C.

```python
brest = sst.sel(lat=..., lon=..., method="nearest")
brest.plot()
mois_brest = brest.groupby(...).mean()
print(mois_brest.idxmin().item(), mois_brest.idxmax().item(), ...)
nino34 = sst.sel(lat=slice(..., ...), lon=slice(..., ...)).mean(...)
print(nino34.sizes)
```

### Solution

In [ ]:
brest = sst.sel(lat=48.4, lon=360 - 4.5, method="nearest")
print(float(brest.lat), float(brest.lon))
brest.plot()
mois_brest = brest.groupby("time.month").mean()
print(mois_brest.round(1).values)
ecart = float(mois_brest.max() - mois_brest.min())
print(mois_brest.idxmin().item(), mois_brest.idxmax().item(), round(ecart, 1))
nino34 = sst.sel(lat=slice(5, -5), lon=slice(190, 240)).mean(["lat", "lon"])
print(nino34.sizes)
# La maille la plus proche de Brest est centrée sur 48° N, 356° E (4° O).
# Le mois le plus froid est mars (10,2 °C), le plus chaud août (18,1 °C) : 7,9 °C d'écart.
# nino34 a 624 valeurs, une par mois.

## 3. Climatologie et anomalies

La température de la région Niño 3.4 suit un cycle annuel : pour comparer une année à une autre, il faut d'abord retirer ce cycle. Calculez :

- la *climatologie* de `nino34` : sa moyenne de chaque mois de l'année (12 valeurs), sur la période de référence 1991-2020 ;
- les *anomalies* `anomalies_nino` : l'écart de chaque mois de `nino34` à la climatologie de son mois (624 valeurs) ;
- les cinq mois aux anomalies les plus fortes : de quelles années sont-ils ?
- l'indice ONI de la NOAA (*Oceanic Niño Index*), la moyenne glissante des anomalies sur trois mois, centrée. La NOAA déclare un épisode El Niño quand il dépasse +0,5 °C cinq mois de suite, La Niña quand il passe sous −0,5 °C. Tracez les anomalies et l'ONI sur un même graphique, avec les deux seuils : quels ont été les trois épisodes El Niño les plus forts ?

**Documentation utile**

- [`DataArray.sortby`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sortby.html)
- [`DataArray.rolling`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.rolling.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

La climatologie : `sel(time=slice("1991", "2020"))`, puis `groupby("time.month").mean()`. La soustraction `nino34.groupby("time.month") - climatologie` retire à chaque mois la climatologie de son mois. `anomalies_nino.sortby(anomalies_nino, ascending=False)` trie les mois de la plus forte anomalie à la plus faible, `isel(time=slice(0, 5))` garde les cinq premiers, et `.time.dt.strftime("%Y-%m")` affiche leurs dates lisiblement. `rolling(time=3, center=True).mean()` calcule l'ONI ; `oni.groupby("time.year").max()`, son maximum de chaque année, que l'on trie comme les anomalies. Un épisode s'étend souvent sur deux années. `plt.axhline(0.5)` trace un seuil. L'anomalie la plus forte dépasse +2,5 °C.

```python
climatologie = nino34.sel(time=...).groupby(...).mean()
anomalies_nino = nino34.groupby(...) - climatologie
fortes = anomalies_nino.sortby(...).isel(time=slice(0, 5))
print(fortes.time.dt.strftime("%Y-%m").values, fortes.round(2).values)
oni = anomalies_nino.rolling(...).mean()
max_par_an = oni.groupby("time.year").max()
print(max_par_an.sortby(...).isel(year=slice(0, 6)).values)
anomalies_nino.plot(color="lightgray", label="Anomalie mensuelle")
oni.plot(label="ONI")
plt.axhline(...)
plt.axhline(...)
plt.legend()
```

### Solution

In [ ]:
climatologie = nino34.sel(time=slice("1991", "2020")).groupby("time.month").mean()
anomalies_nino = nino34.groupby("time.month") - climatologie
print(climatologie.round(2).values)
fortes = anomalies_nino.sortby(anomalies_nino, ascending=False).isel(time=slice(0, 5))
print(fortes.time.dt.strftime("%Y-%m").values, fortes.round(2).values)
oni = anomalies_nino.rolling(time=3, center=True).mean()
max_par_an = oni.groupby("time.year").max()
top = max_par_an.sortby(max_par_an, ascending=False).isel(year=slice(0, 6))
print(top.year.values, top.round(2).values)
anomalies_nino.plot(color="lightgray", label="Anomalie mensuelle")
oni.plot(label="ONI (moyenne sur 3 mois)")
plt.axhline(0.5, color="red", linestyle="--")
plt.axhline(-0.5, color="blue", linestyle="--")
plt.ylabel("Anomalie Niño 3.4 (°C)")
plt.legend()
# La climatologie culmine en mai (27,9 °C) et descend à 26,6 °C en décembre-janvier.
# Les cinq plus fortes anomalies : novembre 2015 à janvier 2016 (+2,7 °C au plus),
# puis novembre 1997 et janvier 1998.
# Les trois plus forts épisodes El Niño : 2015-2016 (ONI de +2,64 °C), 1997-1998
# (+2,35 °C) et 1982-1983 (+2,15 °C).

## 4. Calculer par blocs, écrire en NetCDF et en Zarr

Rouvrez le fichier avec `xr.tutorial.open_dataset("ersstv5", chunks={"time": 120})` : chaque variable devient un tableau Dask, en blocs de 120 mois, et les calculs deviennent paresseux. Sur la carte entière cette fois, calculez les anomalies de `sst` : la climatologie 1991-2020 de chaque maille, puis l'écart de chaque mois à la climatologie de son mois. Affichez `.data` avant et après ce calcul : rien n'est encore calculé, mais que sont devenus les blocs ?

Regroupez les blocs par 120 mois (`chunk`), nommez le résultat `anomalie` et donnez-lui les attributs `long_name` et `units` qui le décrivent, puis écrivez-le dans `anomalies.nc` (`to_netcdf`) et dans `anomalies.zarr` (`to_zarr`) : c'est l'écriture qui déclenche le calcul, bloc par bloc.

Comparez la taille des deux, et comptez les fichiers du dossier Zarr. Relisez-les (`xr.open_dataset`, `xr.open_zarr`) : ont-ils les mêmes valeurs ? Tracez enfin, à partir du Zarr, la carte des anomalies de décembre 2015.

**Documentation utile**

- [`DataArray.chunk`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.chunk.html)
- [`DataArray.to_netcdf`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.to_netcdf.html)
- [`DataArray.to_zarr`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.to_zarr.html)
- [`xarray.open_zarr`](https://docs.xarray.dev/en/stable/generated/xarray.open_zarr.html)
- [`pathlib.Path.rglob`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.rglob)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le calcul des anomalies est celui de l'exercice 3, sur `sst` entière au lieu de `nino34`. La soustraction par `groupby` découpe le tableau en blocs d'un mois : 624 blocs, qui feraient 624 fichiers Zarr. Elle ajoute aussi une coordonnée `month`, que `drop_vars("month")` retire. `.chunk({"time": 120})` regroupe les blocs, `.rename("anomalie")` nomme le tableau, et `anomalies.attrs = {...}` remplace les attributs hérités de `sst`, qui décriraient une température. `to_zarr(..., mode="w")` remplace un dossier existant, si vous relancez la cellule ; `consolidated=False`, à l'écriture comme à la lecture, évite un avertissement : le format Zarr 3 ne prévoit pas de métadonnées consolidées. `Path(chemin).stat().st_size` donne la taille d'un fichier, en octets ; celle d'un dossier est la somme de celles des fichiers que `rglob("*")` y trouve (`is_file()`). `identical` compare les valeurs, les coordonnées et les attributs.

```python
ds_blocs = xr.tutorial.open_dataset("ersstv5", chunks=...)
sst_blocs = ds_blocs.sst
print(sst_blocs.data)
climatologie_cartes = sst_blocs.sel(...).groupby(...).mean()
anomalies = (sst_blocs.groupby(...) - climatologie_cartes).drop_vars("month")
print(anomalies.data)
anomalies = anomalies.chunk(...).rename(...)
anomalies.attrs = {"long_name": ..., "units": ...}
anomalies.to_netcdf(...)
anomalies.to_zarr(..., mode="w", consolidated=False)
taille_nc = ...
fichiers_zarr = [f for f in Path(...).rglob("*") if f.is_file()]
taille_zarr = ...
print(taille_nc, taille_zarr, len(fichiers_zarr))
relu_nc = xr.open_dataset(...)
relu_zarr = xr.open_zarr(..., consolidated=False)
print(relu_nc.anomalie.identical(...))
relu_zarr.anomalie.sel(...).plot(cmap="RdBu_r")
```

### Solution

In [ ]:
ds_blocs = xr.tutorial.open_dataset("ersstv5", chunks={"time": 120})
sst_blocs = ds_blocs.sst
print(sst_blocs.data)
reference = sst_blocs.sel(time=slice("1991", "2020"))
climatologie_cartes = reference.groupby("time.month").mean()
anomalies = (sst_blocs.groupby("time.month") - climatologie_cartes).drop_vars("month")
print(anomalies.data)
anomalies = anomalies.chunk({"time": 120}).rename("anomalie")
anomalies.attrs = {
  "long_name": "Anomalie de température de surface de la mer (base 1991-2020)",
  "units": "degC",
}
anomalies.to_netcdf("anomalies.nc")
anomalies.to_zarr("anomalies.zarr", mode="w", consolidated=False)
taille_nc = Path("anomalies.nc").stat().st_size
fichiers_zarr = [f for f in Path("anomalies.zarr").rglob("*") if f.is_file()]
taille_zarr = sum(f.stat().st_size for f in fichiers_zarr)
print(f"NetCDF : {taille_nc / 1e6:.1f} Mo")
print(f"Zarr : {taille_zarr / 1e6:.1f} Mo en {len(fichiers_zarr)} fichiers")
relu_nc = xr.open_dataset("anomalies.nc")
relu_zarr = xr.open_zarr("anomalies.zarr", consolidated=False)
print(relu_zarr.anomalie.data)
print(relu_nc.anomalie.identical(relu_zarr.anomalie))
relu_zarr.anomalie.sel(time="2015-12").plot(cmap="RdBu_r")
# Après la soustraction, les blocs ne font plus qu'un mois : (1, 89, 180), 624 blocs.
# Le NetCDF pèse 40 Mo : xarray ne le compresse pas sans encoding (zlib). Le Zarr pèse
# 21 Mo, compressé par défaut (Zstandard), en 14 fichiers : 6 blocs de 120 mois pour
# anomalie, un par coordonnée, et les métadonnées (zarr.json).
# Les deux relectures sont identiques, attributs compris.
# Décembre 2015 : une langue chaude, jusqu'à +3,6 °C, le long de l'équateur, de la
# côte de l'Amérique du Sud au centre du Pacifique.

## 5. Sous le NetCDF, du HDF5

Un fichier NetCDF-4 est un fichier HDF5 qui suit des conventions. Ouvrez `anomalies.nc` avec `h5py`, sans xarray :

- quels objets contient-il ?
- quelle forme, quel type et quels attributs a le jeu de données `anomalie` ?
- que valent les trois premières valeurs de `time` dans le fichier, et son attribut `units` ? Qu'en fait xarray ?
- lisez la valeur de décembre 2015 (indice 551 du temps) sur l'équateur, à la longitude 250 (indices 44 et 125), et comparez-la à celle que donne xarray, `relu_nc.anomalie.sel(time="2015-12", lat=0, lon=250)`.

Quel travail xarray fait-il donc pour vous à la lecture ?

**Documentation utile**

- [Démarrer avec h5py](https://docs.h5py.org/en/stable/quick.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`with h5py.File("anomalies.nc") as fichier:` ouvre le fichier en lecture et le referme à la fin du bloc. Un fichier HDF5 se parcourt comme un dictionnaire : `list(fichier)` donne ses noms, `fichier["anomalie"]` un jeu de données, qui a `.shape`, `.dtype` et `.attrs` (un dictionnaire, dont `list` donne les noms). Indexer un jeu de données, `jeu[551, 44, 125]`, ne lit sur le disque que ce qu'on demande. `DIMENSION_LIST` relie chaque axe de `anomalie` au jeu de données de sa coordonnée ; les attributs `_Netcdf4…` sont des conventions internes de NetCDF.

```python
with h5py.File("anomalies.nc") as fichier:
  print(list(fichier))
  jeu = fichier["anomalie"]
  print(jeu.shape, jeu.dtype, list(jeu.attrs))
  print(fichier["time"][:3], fichier["time"].attrs[...])
  print(jeu[..., ..., ...])
print(relu_nc.time.values[:3])
print(relu_nc.anomalie.sel(...).item())
```

### Solution

In [ ]:
with h5py.File("anomalies.nc") as fichier:
  print(list(fichier))
  jeu = fichier["anomalie"]
  print(jeu.shape, jeu.dtype, list(jeu.attrs))
  print(jeu.attrs["long_name"], jeu.attrs["units"])
  print(fichier["time"][:3], fichier["time"].attrs["units"])
  print(jeu[551, 44, 125])
print(relu_nc.time.values[:3])
print(relu_nc.anomalie.sel(time="2015-12", lat=0, lon=250).item())
# Quatre jeux de données : anomalie et ses trois coordonnées, lat, lon et time.
# anomalie : (624, 89, 180), float32 ; ses attributs sont les nôtres (long_name, units),
# _FillValue (NaN, la valeur manquante) et ceux de NetCDF (DIMENSION_LIST…).
# Dans le fichier, time vaut 62091, 62122, 62150 : des jours depuis le 1er janvier 1800,
# que xarray décode en dates (1970-01-01, 1970-02-01, 1970-03-01).
# Les deux lectures donnent la même valeur, +3,46 °C.
# xarray nomme les axes et leur attache leurs coordonnées, décode les dates et les
# valeurs manquantes, et permet de sélectionner par étiquette plutôt que par indice.

## 6. Facultatif : animer les anomalies de 2015 et 2016

Faites défiler, mois par mois, les cartes d'anomalies de janvier 2015 à décembre 2016, l'épisode El Niño le plus fort de la série. Avec `matplotlib.animation.FuncAnimation`, une fonction redessine la figure pour chaque image ; `HTML(animation.to_jshtml())` affiche l'animation dans le notebook, avec ses boutons de lecture.

Gardez la même échelle de couleurs pour toutes les images (`vmin`, `vmax`) : pourquoi est-ce indispensable ? Où l'anomalie chaude apparaît-elle, comment s'étend-elle, et que devient-elle en 2016 ?

**Documentation utile**

- [`matplotlib.animation.FuncAnimation`](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FuncAnimation.html)
- [`Animation.to_jshtml`](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.Animation.html#matplotlib.animation.Animation.to_jshtml)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Chargez d'abord les 24 cartes en mémoire (`.load()`), pour ne pas relire le Zarr à chaque image. Tracez la première avec `.plot(ax=ax, cmap="RdBu_r", vmin=-4, vmax=4)` : elle renvoie l'objet dessiné (un `QuadMesh`), dont la fonction de l'animation remplace les valeurs, `image.set_array(cartes.isel(time=i).values.ravel())`, avant de changer le titre (`ax.set_title`). `FuncAnimation(fig, fonction, frames=24, interval=300)` appelle la fonction pour chaque image, une toutes les 300 ms. `plt.close(fig)` évite que la figure fixe s'affiche en plus de l'animation.

```python
cartes = relu_zarr.anomalie.sel(time=slice(..., ...)).load()
fig, ax = plt.subplots(figsize=(8, 4))
image = cartes.isel(time=0).plot(ax=ax, cmap=..., vmin=..., vmax=...)


def dessiner(i):
  image.set_array(...)
  ax.set_title(...)
  return [image]


animation = FuncAnimation(fig, dessiner, frames=..., interval=300)
plt.close(fig)
HTML(animation.to_jshtml())
```

### Solution

In [ ]:
cartes = relu_zarr.anomalie.sel(time=slice("2015-01", "2016-12")).load()
fig, ax = plt.subplots(figsize=(8, 4))
image = cartes.isel(time=0).plot(ax=ax, cmap="RdBu_r", vmin=-4, vmax=4)


def dessiner(i):
  image.set_array(cartes.isel(time=i).values.ravel())
  ax.set_title(str(cartes.time.values[i])[:7])
  return [image]


animation = FuncAnimation(fig, dessiner, frames=cartes.sizes["time"], interval=300)
plt.close(fig)
# Sans échelle fixe, chaque image aurait la sienne : une même couleur n'y voudrait pas
# le même nombre de degrés, et la croissance de l'anomalie ne se verrait pas.
# L'anomalie chaude naît au printemps 2015 près des côtes de l'Équateur et du Pérou,
# s'étire vers l'ouest le long de l'équateur jusqu'au centre du Pacifique, culmine
# à la fin de 2015, puis se résorbe au printemps 2016 ; à la fin de 2016, l'est du
# Pacifique équatorial est plus froid que la normale : une faible Niña.
HTML(animation.to_jshtml())